# House Stage 1 regression: fixed incumbency coefficient, rolling 2018–2022

The classic formula with the `inc_signed` coefficient **imposed** rather than estimated, against unrestricted OLS, in the same expanding-window backtest as `01_regression_2018to2022_eval.ipynb`. Each window refits every model on prior elections only and scores the next one:

| eval year | train years |
|---|---|
| 2018 | 2006–2016 |
| 2020 | 2006–2018 |
| 2022 | 2006–2020 |

For a fixed coefficient *c*, `run_ols_fixed` (`experiments/house/run.py`) fits OLS of `adj_margin_2p_norm − c · inc_signed` on the other terms, re-estimating all of them, then adds `c · inc_signed` back to the fitted values and predictions. The predictions are scored against `adj_margin_2p_norm` as usual. Every model in a window uses the same training and evaluation rows, and the evaluation subsets (valid, `COMPETITIVE`) are the original notebook's. 2024 is never used (`USE_HOLDOUT = False`).

In [18]:
%load_ext autoreload
%autoreload 2
import sys
from pathlib import Path

import pandas as pd
from IPython.display import Markdown

# The repository root is the first folder up from here that holds pyproject.toml, so this works at any depth
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT / "src"))
sys.path.insert(0, str(ROOT))  # repository root, for experiments.house
import house.regression as hr  # hr.<Tab> lists everything; names are in src/house/regression/__init__.py
from experiments.house import evaluation, tracking
from experiments.house.run import run_ols, run_ols_fixed

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [19]:
import warnings

with warnings.catch_warnings(action="ignore"):  # loading warns with the same summaries printed below
    predictors = hr.load_predictors()  # data/processed/house/modeling.csv, every year (2006-2026)
    targets = hr.load_targets()        # data/processed/house/targets.csv, every year (2006-2024)
print(hr.validate_schema(predictors).summary())  # .issues lists every affected key

In [20]:
FORMULA = "adj_margin_2p_norm ~ gen_margin_raw + partisan_lean + inc_signed"
ROLLING_WINDOWS = {  # eval year: its expanding training window
    2018: [2006, 2008, 2010, 2012, 2014, 2016],
    2020: [2006, 2008, 2010, 2012, 2014, 2016, 2018],
    2022: [2006, 2008, 2010, 2012, 2014, 2016, 2018, 2020],
}
USE_HOLDOUT = False     # never True here: no window touches HOLDOUT_YEARS (2024)
REQUIRE_COMPLETE = []   # extra columns a row must have; every model below uses FORMULA's columns, so the same rows
# Evaluation only (training is unaffected): observed, non-imputed targets of these race types are scored ("valid")
EVAL_RACE_TYPES = ["dr", "dr_multi", "dr_jungle", "dr_rcv", "dr_writein"]
# Competitive subsets of the valid targets: name -> (column, threshold); a row is in one when abs(column) <= threshold.
# column is a column name or arithmetic of them, from the targets or the predictors
EX_ANTE_THRESHOLD = 10   # points: any non-negative int or float, e.g. 5, 7, or 7.5
EX_POST_THRESHOLD = 7.5
COMPETITIVE = {
    # expected margin under uniform swing: two-party lean (2 x cook_pvi) + eve-of-election two-party generic ballot
    "competitive_ex_ante": ("partisan_lean + gb_margin_2p_norm", EX_ANTE_THRESHOLD),
    "competitive_ex_post": ("margin_2p_norm", EX_POST_THRESHOLD),  # observed two-party margin
}
SUMMARY_SECTIONS = {"ALL": "valid", "EX_ANTE_COMPETITIVE": "competitive_ex_ante",
                    "EX_POST_COMPETITIVE": "competitive_ex_post"}
RANK_BY = "mae"     # variants ranked by mean out-of-sample MAE ...
TIEBREAK = "rmse"   # ... ties broken by RMSE

## Fixed incumbency coefficients

`inc_signed` is +1 for a Democratic incumbent, −1 for a Republican one and 0 for an open seat, so its coefficient is the incumbency advantage in points of D margin. Each value in `FIXED_INC_COEFFICIENTS` is imposed in every window. Unrestricted OLS (`ols`) estimates it freely and is the base of every Δ below. Fixing it at 0 is the same as dropping the term.

In [21]:
FIXED_TERM = "inc_signed"
FIXED_INC_COEFFICIENTS = [0, 2.5, 5.0, 7.5, 10.0]  # imposed coefficients, points of D margin

FIXED_VARIANTS = {f"inc_fixed_{c:g}": c for c in FIXED_INC_COEFFICIENTS}  # variant: its imposed coefficient
list(FIXED_VARIANTS)

['inc_fixed_0',
 'inc_fixed_2.5',
 'inc_fixed_5',
 'inc_fixed_7.5',
 'inc_fixed_10']

In [22]:
# Every window: unrestricted OLS (the base), then each fixed coefficient, all on the same rows
rolling = {}
for eval_year, train_years in ROLLING_WINDOWS.items():
    ols = run_ols(
        formula=FORMULA,
        train_years=train_years,
        eval_years=[eval_year],
        predict_years=[],
        predictors=predictors,
        targets=targets,
        use_holdout=USE_HOLDOUT,
        require_complete=REQUIRE_COMPLETE,
        race_types=EVAL_RACE_TYPES,
        competitive=COMPETITIVE,
        name="ols",
        save=False,
    )
    fixed = [
        run_ols_fixed(
            formula=FORMULA,
            train_years=train_years,
            eval_years=[eval_year],
            predict_years=[],
            fixed={FIXED_TERM: c},
            predictors=predictors,
            targets=targets,
            use_holdout=USE_HOLDOUT,
            require_complete=REQUIRE_COMPLETE,
            race_types=EVAL_RACE_TYPES,
            competitive=COMPETITIVE,
            name=name,
            save=False,
        )
        for name, c in FIXED_VARIANTS.items()
    ]
    rolling[eval_year] = {"ols_vs_fixed": [ols, *fixed]}  # one block, OLS first: the base of every delta

/Users/ElijahSong/Documents/Election26/experiments/house/run.py:144: UserWarning: 2 predictions outside +-100 (not clipped): (2018, 'NY', 13), (2018, 'NY', 15)
  return _fit_and_score(
/Users/ElijahSong/Documents/Election26/experiments/house/run.py:144: UserWarning: 2 predictions outside +-100 (not clipped): (2018, 'NY', 13), (2018, 'NY', 15)
  return _fit_and_score(
/Users/ElijahSong/Documents/Election26/experiments/house/run.py:144: UserWarning: 1 predictions outside +-100 (not clipped): (2018, 'NY', 15)
  return _fit_and_score(
/Users/ElijahSong/Documents/Election26/experiments/house/run.py:144: UserWarning: 2 predictions outside +-100 (not clipped): (2020, 'NY', 13), (2020, 'NY', 15)
  return _fit_and_score(


In [23]:
# the regression each variant actually fits (the same in every window)
pd.Series({run.config["name"]: run.config.get("fit_formula", run.config["formula"])
           for run in rolling[next(iter(ROLLING_WINDOWS))]["ols_vs_fixed"]}, name="fit formula").to_frame()

,fit formula
ols,adj_margin_2p_norm ~ gen_margin_raw + partisan...
inc_fixed_0,I(adj_margin_2p_norm - 0.0 * inc_signed) ~ gen...
inc_fixed_2.5,I(adj_margin_2p_norm - 2.5 * inc_signed) ~ gen...
inc_fixed_5,I(adj_margin_2p_norm - 5.0 * inc_signed) ~ gen...
inc_fixed_7.5,I(adj_margin_2p_norm - 7.5 * inc_signed) ~ gen...
inc_fixed_10,I(adj_margin_2p_norm - 10.0 * inc_signed) ~ ge...


## Results

For each section (ALL, EX_ANTE and EX_POST on the original notebook's definitions), there are two tables. The first is **by eval year**, each window on its own. The second is the **mean over 2018, 2020 and 2022**, each window weighted equally. Variants are ranked by mean MAE, with ties broken by RMSE.

- **Metrics:** `n_train`, `n`, MAE, RMSE, bias (mean residual, actual − predicted: + means Democrats outperformed the forecast), accuracy, precision D/R and misses.
- **Mean table extras:** `|bias|` (the mean of each window's |bias|, so opposite misses don't cancel) and `mae_<year>` for each window.
- **Δ columns:** variant − unrestricted OLS in the same window, on the same rows. ✓ (green) marks an improvement and ✗ (red) a regression: lower MAE, RMSE, |bias| and misses, higher accuracy and precision.

In [24]:
VARIANTS = len(FIXED_VARIANTS)  # show every variant, ranked

for section, subset in SUMMARY_SECTIONS.items():
    by_year = pd.concat(
        {year: tracking.best_of_blocks(blocks, subset=subset, rank_by=RANK_BY, top=VARIANTS, tiebreak=TIEBREAK)[0]
         for year, blocks in rolling.items()}, names=["eval_year"]).droplevel("block")
    mean, _ = tracking.rolling_best_of(rolling, subset=subset, rank_by=RANK_BY, top=VARIANTS, tiebreak=TIEBREAK)
    display(Markdown(f"### {section} (`{subset}`): by eval year"))
    display(tracking.style_best_of(by_year))
    display(Markdown(f"### {section} (`{subset}`): mean of {', '.join(map(str, ROLLING_WINDOWS))}, "
                     f"ranked by {RANK_BY} (ties: {TIEBREAK})"))
    display(tracking.style_best_of(mean.droplevel("block")))

### ALL (`valid`): by eval year

### ALL (`valid`): mean of 2018, 2020, 2022, ranked by mae (ties: rmse)

,n_train,n,mae,rmse,bias,|bias|,accuracy_pct,precision_D_pct,precision_R_pct,missed_races,missed_pct,mae_2018,mae_2020,mae_2022,Δmae,Δrmse,Δaccuracy_pct,Δprecision_D_pct,Δprecision_R_pct,Δmissed_races,Δmissed_pct,Δ|bias|
run,,,,,,,,,,,,,,,,,,,,,,
ols,2680,400,7.31,8.91,-1.03,2.59,94.89,96.35,94.22,20,5.11,7.72,7.93,6.27,-,-,-,-,-,-,-,-
inc_fixed_2.5,2680,400,5.69,7.31,-0.48,1.91,94.65,96.14,93.44,21,5.35,6.16,5.97,4.93,-1.62 ✓,-1.60 ✓,-0.24 ✗,-0.21 ✗,-0.78 ✗,+1 ✗,+0.24 ✗,-0.68 ✓
inc_fixed_5,2680,400,5.71,7.26,-0.61,2.08,94.89,96.04,94.16,20,5.11,6.09,6.02,5.02,-1.59 ✓,-1.65 ✓,+0.00 ✓,-0.31 ✗,-0.06 ✗,+0,-0.00 ✓,-0.52 ✓
inc_fixed_0,2680,400,5.92,7.67,-0.35,1.74,94.25,95.94,92.77,23,5.75,6.37,6.26,5.12,-1.39 ✓,-1.23 ✓,-0.64 ✗,-0.42 ✗,-1.44 ✗,+3 ✗,+0.64 ✗,-0.85 ✓
inc_fixed_7.5,2680,400,5.99,7.51,-0.75,2.24,95.06,96.28,94.39,20,4.94,6.27,6.40,5.31,-1.31 ✓,-1.39 ✓,+0.17 ✓,-0.08 ✗,+0.17 ✓,-1 ✓,-0.17 ✓,-0.35 ✓
inc_fixed_10,2680,400,6.52,8.05,-0.88,2.41,94.98,96.15,94.43,20,5.02,6.72,7.04,5.80,-0.79 ✓,-0.85 ✓,+0.09 ✓,-0.20 ✗,+0.21 ✓,-0 ✓,-0.09 ✓,-0.19 ✓


### EX_ANTE_COMPETITIVE (`competitive_ex_ante`): by eval year

### EX_ANTE_COMPETITIVE (`competitive_ex_ante`): mean of 2018, 2020, 2022, ranked by mae (ties: rmse)

,n_train,n,mae,rmse,bias,|bias|,accuracy_pct,precision_D_pct,precision_R_pct,missed_races,missed_pct,mae_2018,mae_2020,mae_2022,Δmae,Δrmse,Δaccuracy_pct,Δprecision_D_pct,Δprecision_R_pct,Δmissed_races,Δmissed_pct,Δ|bias|
run,,,,,,,,,,,,,,,,,,,,,,
ols,2680,91,7.93,9.24,+0.91,4.18,82.90,88.14,83.16,16,17.10,9.47,7.95,6.36,-,-,-,-,-,-,-,-
inc_fixed_2.5,2680,91,5.28,6.56,+1.17,1.88,80.37,85.68,76.71,18,19.63,5.57,4.89,5.37,-2.65 ✓,-2.68 ✓,-2.53 ✗,-2.46 ✗,-6.45 ✗,+2 ✗,+2.53 ✗,-2.30 ✓
inc_fixed_5,2680,91,5.30,6.41,+1.06,2.23,81.64,85.30,79.99,17,18.36,6.01,4.76,5.14,-2.62 ✓,-2.83 ✓,-1.25 ✗,-2.84 ✗,-3.17 ✗,+1 ✗,+1.25 ✗,-1.95 ✓
inc_fixed_7.5,2680,91,5.74,6.85,+0.95,2.69,82.47,88.83,81.71,16,17.53,6.67,5.30,5.26,-2.18 ✓,-2.39 ✓,-0.43 ✗,+0.69 ✓,-1.45 ✗,+0 ✗,+0.43 ✗,-1.49 ✓
inc_fixed_0,2680,91,5.76,7.24,+1.28,1.52,78.55,83.95,74.65,19,21.45,5.42,5.74,6.13,-2.16 ✓,-2.00 ✓,-4.35 ✗,-4.18 ✗,-8.51 ✗,+4 ✗,+4.35 ✗,-2.66 ✓
inc_fixed_10,2680,91,6.56,7.79,+0.85,3.37,82.90,88.14,83.16,16,17.10,7.60,6.36,5.74,-1.36 ✓,-1.45 ✓,+0.00,+0.00,+0.00,+0,+0.00,-0.81 ✓


### EX_POST_COMPETITIVE (`competitive_ex_post`): by eval year

### EX_POST_COMPETITIVE (`competitive_ex_post`): mean of 2018, 2020, 2022, ranked by mae (ties: rmse)

,n_train,n,mae,rmse,bias,|bias|,accuracy_pct,precision_D_pct,precision_R_pct,missed_races,missed_pct,mae_2018,mae_2020,mae_2022,Δmae,Δrmse,Δaccuracy_pct,Δprecision_D_pct,Δprecision_R_pct,Δmissed_races,Δmissed_pct,Δ|bias|
run,,,,,,,,,,,,,,,,,,,,,,
ols,2680,60,10.56,12.20,+0.86,6.55,69.93,80.00,74.20,18,30.07,12.25,11.88,7.54,-,-,-,-,-,-,-,-
inc_fixed_0,2680,60,6.41,8.42,+2.29,2.29,68.37,73.10,65.97,19,31.63,7.49,5.81,5.93,-4.15 ✓,-3.78 ✓,-1.56 ✗,-6.90 ✗,-8.23 ✗,+1 ✗,+1.56 ✗,-4.26 ✓
inc_fixed_2.5,2680,60,6.43,8.27,+1.97,2.74,68.90,73.51,67.60,19,31.10,8.05,5.62,5.63,-4.12 ✓,-3.93 ✓,-1.04 ✗,-6.49 ✗,-6.60 ✗,+0 ✗,+1.04 ✗,-3.81 ✓
inc_fixed_5,2680,60,6.92,8.63,+1.64,3.53,69.06,71.88,71.00,19,30.94,8.84,6.17,5.77,-3.63 ✓,-3.57 ✓,-0.87 ✗,-8.12 ✗,-3.20 ✗,+0 ✗,+0.87 ✗,-3.02 ✓
inc_fixed_7.5,2680,60,7.80,9.45,+1.32,4.33,69.16,75.49,71.78,19,30.84,9.71,7.49,6.19,-2.76 ✓,-2.75 ✓,-0.77 ✗,-4.51 ✗,-2.42 ✗,+0 ✗,+0.77 ✗,-2.22 ✓
inc_fixed_10,2680,60,8.99,10.63,+0.99,5.20,69.35,75.47,73.08,19,30.65,10.68,9.40,6.89,-1.57 ✓,-1.57 ✓,-0.58 ✗,-4.53 ✗,-1.12 ✗,+0 ✗,+0.58 ✗,-1.35 ✓


## Coefficients

Each model's coefficients by window, then averaged over the three. For the fixed variants, `inc_signed` is the imposed value (it has no standard error) and the other terms are re-estimated around it. Unrestricted OLS's estimate is the `ols` row.

In [25]:
# coefficients by window (rows: eval year and model, columns: terms); inc_signed is imposed for the fixed variants
pd.concat({eval_year: pd.DataFrame({run.config["name"]: run.coefficients["coef"] for run in blocks["ols_vs_fixed"]}).T
           for eval_year, blocks in rolling.items()}, names=["eval_year", "model"]).round(3)

term                     Intercept  gen_margin_raw  partisan_lean  inc_signed
eval_year model                                                              
2018      ols               -0.179           0.875          0.807      13.811
          inc_fixed_0       -0.219           0.770          1.124       0.000
          inc_fixed_2.5     -0.212           0.789          1.067       2.500
          inc_fixed_5       -0.205           0.808          1.010       5.000
          inc_fixed_7.5     -0.198           0.827          0.952       7.500
          inc_fixed_10      -0.190           0.846          0.895      10.000
2020      ols               -0.021           0.892          0.809      12.798
          inc_fixed_0       -0.071           0.792          1.099       0.000
          inc_fixed_2.5     -0.061           0.811          1.042       2.500
          inc_fixed_5       -0.051           0.831          0.986       5.000
          inc_fixed_7.5     -0.042           0.850          0.929       7.500
          inc_fixed_10      -0.032           0.870          0.872      10.000
2022      ols               -0.483           0.827          0.817      11.718
          inc_fixed_0       -0.342           0.756          1.079       0.000
          inc_fixed_2.5     -0.372           0.772          1.023       2.500
          inc_fixed_5       -0.402           0.787          0.967       5.000
          inc_fixed_7.5     -0.432           0.802          0.911       7.500
          inc_fixed_10      -0.462           0.817          0.856      10.000

In [26]:
# each model's coefficients averaged over the windows
_, mean_coefficients = tracking.rolling_best_of(rolling, rank_by=RANK_BY, top=VARIANTS, tiebreak=TIEBREAK)
mean_coefficients.droplevel("block").round(3)

,Intercept,gen_margin_raw,partisan_lean,inc_signed
run,,,,
ols,-0.228,0.865,0.811,12.776
inc_fixed_2.5,-0.215,0.791,1.044,2.500
inc_fixed_5,-0.219,0.809,0.988,5.000
inc_fixed_0,-0.211,0.773,1.101,0.000
inc_fixed_7.5,-0.224,0.826,0.931,7.500
inc_fixed_10,-0.228,0.844,0.874,10.000


## Ablations with incumbency fixed

The ablation blocks of `01_regression_2018to2022_eval.ipynb`, refit with `inc_signed` fixed at `ABLATION_FIXED_INC` (2.5) instead of estimated: demographics, interactions, FEC fundraising, small-dollar fundraising, and the relative economic factors. Each model is `run_ols_fixed(..., fixed={"inc_signed": ABLATION_FIXED_INC})`, so every other coefficient, including each block's added terms, is re-estimated around the fixed incumbency effect. Each block is refit in every window (`ROLLING_WINDOWS`) on prior elections only.

- **Bases:** each block's base is the classic formula with incumbency fixed, and its `Δ` columns compare against that base, on the same rows. A block shares its rows across its models through `require_complete`, every predictor column any of its formulas uses. A block's base is shown separately when its rows differ from the first block's (FEC leaves out the districts it has no data for; small-dollar trains from 2008).
- **Left out:** variants that estimate incumbency freely (`no_inc_signed`, `inc_categorical`, `plus_fundraising_no_inc`) conflict with a fixed coefficient. Fixing it at 0 is the `inc_fixed_0` row above.
- **Interactions** such as `partisan_lean:inc_signed` are estimated on top of the fixed main effect.
- **Economic factors** are the relative indicators (region or state against the nation), each signed by the president's party (`pres_party_signed:<variable>`, as in the original notebook). β is then the effect on the president's party's margin.

The tables are by eval year and mean over the windows, with the same metrics, Δ and ✓ / ✗ marks as above. `ABLATION_TOP` limits each block to its best variants by mean MAE (ties: RMSE). `None` shows every variant.

In [27]:
ABLATION_FIXED_INC = 2.5  # inc_signed imposed in every ablation, points of D margin
ABLATION_TOP = None       # variants shown per block, ranked; None shows all, 2 the top two
ABLATION_BASE = f"inc_fixed_{ABLATION_FIXED_INC:g}"
TARGET = "adj_margin_2p_norm"

# The party holding the White House at each election: +1 Democratic, -1 Republican (the economic factors' sign)
PRES_PARTY_SIGNED = {2006: -1, 2008: -1, 2010: 1, 2012: 1, 2014: 1, 2016: 1, 2018: -1, 2020: -1, 2022: 1, 2024: 1,
                     2026: -1}
predictors = predictors.assign(pres_party_signed=predictors["year"].map(PRES_PARTY_SIGNED))

DEMOGRAPHIC_ABLATIONS = {
    ABLATION_BASE: FORMULA,  # the base
    "plus_acs": FORMULA + " + college + hispanic",
}
INTERACTION_ABLATIONS = {
    ABLATION_BASE: FORMULA,
    "lean_x_inc": FORMULA + " + partisan_lean:inc_signed",
}
FEC_ABLATIONS = {
    ABLATION_BASE: FORMULA,
    "plus_fundraising": FORMULA + " + fundraising_advantage",
    "plus_spending": FORMULA + " + spending_advantage",
    "plus_fundraising_spending": FORMULA + " + fundraising_advantage + spending_advantage",
}
SMALL_DOLLAR_ABLATIONS = {  # trained from 2008: small_dollar_share_diff has no 2006 data
    f"{ABLATION_BASE}_from_2008": FORMULA,
    "plus_fundraising_from_2008": FORMULA + " + fundraising_advantage",
    "plus_fundraising_small_dollar": FORMULA + " + fundraising_advantage + small_dollar_share_diff",
}
RELATIVE_ECON_ABLATIONS = {  # the region or state against the nation, signed by the president's party
    ABLATION_BASE: FORMULA,
    "pres_x_cpi_region_minus_us_yoy": FORMULA + " + pres_party_signed:cpi_region_minus_us_yoy",  # pp, inflation
    "pres_x_gas_region_minus_us": FORMULA + " + pres_party_signed:gas_region_minus_us",          # $ gap, gas price
    "pres_x_log_rel_gas_price": FORMULA + " + pres_party_signed:log_rel_gas_price",              # log ratio, gas
    "pres_x_gas_region_minus_us_yoy": FORMULA + " + pres_party_signed:gas_region_minus_us_yoy",  # relative trend
    "pres_x_unemp_state_minus_us": FORMULA + " + pres_party_signed:unemp_state_minus_us",        # pp, unemployment
    "pres_x_unemp_change_12m_minus_us": FORMULA + " + pres_party_signed:unemp_change_12m_minus_us",  # relative trend
}

ABLATION_BLOCKS = {  # block: (formulas, base first; first training year)
    "demographics": (DEMOGRAPHIC_ABLATIONS, 2006),
    "interactions": (INTERACTION_ABLATIONS, 2006),
    "fec": (FEC_ABLATIONS, 2006),
    "small_dollar": (SMALL_DOLLAR_ABLATIONS, 2008),
    "relative_econ": (RELATIVE_ECON_ABLATIONS, 2006),
}
# each block's shared rows: every predictor column any of its formulas uses
ABLATION_SHARED_COLUMNS = {block: evaluation.shared_columns(formulas=formulas.values(), columns=predictors.columns)
                           for block, (formulas, _) in ABLATION_BLOCKS.items()}
ABLATION_SHARED_COLUMNS

{'demographics': ['gen_margin_raw',
  'partisan_lean',
  'inc_signed',
  'college',
  'hispanic'],
 'interactions': ['gen_margin_raw', 'partisan_lean', 'inc_signed'],
 'fec': ['gen_margin_raw',
  'partisan_lean',
  'inc_signed',
  'fundraising_advantage',
  'spending_advantage'],
 'small_dollar': ['gen_margin_raw',
  'partisan_lean',
  'inc_signed',
  'fundraising_advantage',
  'small_dollar_share_diff'],
 'relative_econ': ['gen_margin_raw',
  'partisan_lean',
  'inc_signed',
  'pres_party_signed',
  'cpi_region_minus_us_yoy',
  'gas_region_minus_us',
  'log_rel_gas_price',
  'gas_region_minus_us_yoy',
  'unemp_state_minus_us',
  'unemp_change_12m_minus_us']}

In [28]:
# the missing-predictor warnings (FEC has no data for a few winners) repeat for every window and model; each run's
# exclusions are in its .excluded and .counts
with warnings.catch_warnings(action="ignore"):
    ablations = {
        eval_year: {
            block: [
                run_ols_fixed(
                    formula=formula,
                    train_years=[year for year in train_years if year >= first_train_year],
                    eval_years=[eval_year],
                    predict_years=[],
                    fixed={FIXED_TERM: ABLATION_FIXED_INC},
                    predictors=predictors,
                    targets=targets,
                    use_holdout=USE_HOLDOUT,
                    require_complete=ABLATION_SHARED_COLUMNS[block],
                    race_types=EVAL_RACE_TYPES,
                    competitive=COMPETITIVE,
                    name=name,
                    save=False,
                )
                for name, formula in formulas.items()
            ]
            for block, (formulas, first_train_year) in ABLATION_BLOCKS.items()
        }
        for eval_year, train_years in ROLLING_WINDOWS.items()
    }

In [29]:
for section, subset in SUMMARY_SECTIONS.items():
    by_year = pd.concat(
        {year: tracking.best_of_blocks(blocks, subset=subset, rank_by=RANK_BY, top=ABLATION_TOP, tiebreak=TIEBREAK)[0]
         for year, blocks in ablations.items()}, names=["eval_year"])
    mean, _ = tracking.rolling_best_of(ablations, subset=subset, rank_by=RANK_BY, top=ABLATION_TOP,
                                       tiebreak=TIEBREAK)
    display(Markdown(f"### Ablations, {section} (`{subset}`), inc_signed fixed at {ABLATION_FIXED_INC:g}: "
                     "by eval year"))
    display(tracking.style_best_of(by_year))
    display(Markdown(f"### Ablations, {section} (`{subset}`), inc_signed fixed at {ABLATION_FIXED_INC:g}: mean of "
                     f"{', '.join(map(str, ROLLING_WINDOWS))}, ranked by {RANK_BY} (ties: {TIEBREAK})"))
    display(tracking.style_best_of(mean))

### Ablations, ALL (`valid`), inc_signed fixed at 2.5: by eval year

### Ablations, ALL (`valid`), inc_signed fixed at 2.5: mean of 2018, 2020, 2022, ranked by mae (ties: rmse)

### Ablations, EX_ANTE_COMPETITIVE (`competitive_ex_ante`), inc_signed fixed at 2.5: by eval year

### Ablations, EX_ANTE_COMPETITIVE (`competitive_ex_ante`), inc_signed fixed at 2.5: mean of 2018, 2020, 2022, ranked by mae (ties: rmse)

### Ablations, EX_POST_COMPETITIVE (`competitive_ex_post`), inc_signed fixed at 2.5: by eval year

### Ablations, EX_POST_COMPETITIVE (`competitive_ex_post`), inc_signed fixed at 2.5: mean of 2018, 2020, 2022, ranked by mae (ties: rmse)

In [30]:
# each ablation's coefficients averaged over the windows; inc_signed is the imposed ABLATION_FIXED_INC throughout.
# The pres_party_signed:<variable> rows are effects on the president's party's margin, per unit of the variable
_, ablation_coefficients = tracking.rolling_best_of(ablations, rank_by=RANK_BY, top=ABLATION_TOP, tiebreak=TIEBREAK)
ablation_coefficients.style.format(precision=3, na_rep="-")